# NumCompute Quickstart

A short tour of each NumCompute module, with runnable examples.

**To run it:**
1. From the project root, run `uv sync` (see the README).
2. Open this notebook and choose the `.venv` Python as the kernel. If VS Code asks to install `ipykernel`, accept.
3. Run all cells from top to bottom. Paths assume the notebook runs from the `demo/` folder.

In [ ]:
from pathlib import Path

import numpy as np

DATA_DIR = Path("..") / "tests" / "test_csv_files"

## 1. Loading data (`io`)

`load_csv` reads a CSV into a 2D float array. Missing values become `NaN`.

In [ ]:
from numcompute.io import load_csv

X = load_csv(DATA_DIR / "missing_data_multiple.csv")
print(X.shape)
X

## 2. Preprocessing (`preprocessing`)

Every transformer has `fit(X)`, `transform(X)` and `fit_transform(X)`. Calling `transform` before `fit` raises `NotFittedError`.

### Filling missing values: `SimpleImputer`

In [ ]:
from numcompute.preprocessing import SimpleImputer

X_filled = SimpleImputer(fill_value=0.0).fit_transform(X)
X_filled

### Scaling: `StandardScaler` and `MinMaxScaler`

`StandardScaler` gives each column mean 0 and standard deviation 1. `MinMaxScaler` maps each column into `feature_range`.

In [ ]:
from numcompute.preprocessing import MinMaxScaler, StandardScaler

X_std = StandardScaler().fit_transform(X_filled)
print("column means:", X_std.mean(axis=0).round(6) + 0.0)
print("column stds: ", X_std.std(axis=0).round(6))

X_minmax = MinMaxScaler(feature_range=(-1, 1)).fit_transform(X_filled)
X_minmax

### Learn on training data, apply to new data

`fit` learns from one dataset; `transform` reuses what was learned on any new data with the same columns.

In [ ]:
scaler = StandardScaler().fit(X_filled)

new_data = np.array([[5.0, 5.0, 5.0]])
scaler.transform(new_data)

### Categorical data: `OneHotEncoder`

Each column becomes one 0/1 column per category (sorted). Unknown categories raise an error unless `handle_unknown="ignore"`.

In [ ]:
from numcompute.preprocessing import OneHotEncoder

colours = [["red", "S"], ["blue", "L"], ["red", "L"]]

encoder = OneHotEncoder().fit(colours)
print(encoder.categories_)
encoder.transform(colours)

In [ ]:
OneHotEncoder(handle_unknown="ignore").fit(colours).transform([["green", "S"]])

## 3. Numerical gradients (`optim`)

`grad` and `jacobian` estimate derivatives with finite differences. `method="central"` (default) is more accurate than `"forward"`.

The gradient of $f(x) = \sum x_i^2$ is $2x$:

In [ ]:
from numcompute.optim import grad, jacobian


def sum_of_squares(x):
    return np.sum(x**2)


x = np.array([1.0, 2.0, 3.0])
print("central:", grad(sum_of_squares, x))
print("forward:", grad(sum_of_squares, x, method="forward"))

For a function with several outputs, `jacobian` returns shape `(outputs, inputs)`. For $F(x) = [x_0 x_1,\ \sin x_0]$ the exact answer is $\begin{bmatrix} x_1 & x_0 \\ \cos x_0 & 0 \end{bmatrix}$:

In [ ]:
def vector_func(x):
    return np.array([x[0] * x[1], np.sin(x[0])])


jacobian(vector_func, [1.0, 2.0])

## 4. Other modules

*Module owners: add a section here for your module (`sort_search`, `rank`, `stats`, `metrics`, `pipeline`), following the pattern above: a short heading, one sentence, then a small runnable example.*